
# Portfolio Health Analysis

**Case:** PH – Portfolio Health

**Your Mission:** Investigate portfolio health crisis at EduFin using SQL.

---

**Situation:** Portfolio crisis confirmed. Board knows something is wrong – they need you to quantify WHAT.

**Your Assignment:** Portfolio health assessment using SQL investigation.

---

# Dataset Progression

### PH – Portfolio Health

Dataset used:

`workspace.edufin_static`

Purpose:

* Build baseline portfolio understanding
* Execute first portfolio assessment case
* No dataset switching in this case

---

# Deliverables: 
- Queries 1A-1E as specified below
- Executive summary interpreting findings
- WHY Gate answers (business reasoning)
- Scale comparison observations (5K vs 500K)

---
# BEFORE YOU START

1. Run Data Validation Check (Cell 2)
2. Solve queries sequentially
3. Use `workspace.edufin_static`
4. Complete all deliverables before submission

---

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.edufin_static;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.edufin_static.loans AS
SELECT *
FROM workspace.edufin_small.loans;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- DATA VALIDATION CHECK
-- Run this cell FIRST to verify your environment is ready
-- This is NOT a graded query - just a system check

SELECT 
  COUNT(*) AS total_loans,
  COUNT(DISTINCT loan_status) AS status_types,
  COUNT(DISTINCT customer_id) AS unique_customers,
  MIN(loan_amount) AS min_loan,
  MAX(loan_amount) AS max_loan
FROM workspace.edufin_static.loans;

-- Expected: ~5000 loans, 4 status types, ~2400 customers
-- If you see 0 loans or errors, contact support before proceeding

total_loans,status_types,unique_customers,min_loan,max_loan
5000,4,3158,50352,1999991


In [0]:
%sql
-- QUERY 1A: Portfolio Status Distribution
--
-- BUSINESS PURPOSE:
-- How is the portfolio distributed across statuses?
-- Are most loans healthy (Active/Closed) or problematic (Defaulted/Overdue)?
-- This establishes the baseline: Is this a localized issue or portfolio-wide crisis?
--
-- WHAT TO CALCULATE:
-- Show how the portfolio is distributed across different statuses.
-- Include both counts and percentages for each status.
-- Sort by largest category first to show the dominant pattern immediately.
--
-- EXPECTED INSIGHT:
-- If Defaulted + Overdue > 15% combined, this is a systemic crisis, not isolated incidents.

-- Write your query below:


In [0]:
%sql
SELECT 
    loan_status,
    COUNT(*) AS loan_count,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM workspace.edufin_static.loans
GROUP BY loan_status
ORDER BY loan_count DESC;

-- I grouped loans by status to determine whether portfolio issues are isolated or widespread.
-- Including percentages allows comparisonof the relative size of each category.

loan_status,loan_count,percentage
Active,2213,44.26
Closed,1263,25.26
Defaulted,1026,20.52
Overdue,498,9.96


In [0]:
%sql
-- QUERY 1B: Portfolio Scale Metrics
--
-- BUSINESS PURPOSE:
-- What's the scale of operations? Is this a ₹50 Cr portfolio or ₹500 Cr portfolio?
-- Are we talking about 5,000 customers or 50,000?
-- Scale determines whether a ₹12 Cr discrepancy is catastrophic or manageable.
--
-- WHAT TO CALCULATE:
-- Establish the scale of operations:
-- How many unique customers do we serve?
-- How many total loans are active in the system?
-- What's the total portfolio value (in Crores)?
-- What's the average loan size (in Lakhs)?
--
-- EXPECTED INSIGHT:
-- If average loan size is ₹5L+ and portfolio is ₹500 Cr+, defaults hitting ₹12 Cr signals high-value customer concentration risk.

-- Write your query below:


In [0]:
%sql
SELECT
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(*) AS total_loans,
    ROUND(SUM(loan_amount) / 10000000, 2) AS total_portfolio_crores,
    ROUND(AVG(loan_amount) / 100000, 2) AS avg_loan_size_lakhs
FROM workspace.edufin_static.loans;

unique_customers,total_loans,total_portfolio_crores,avg_loan_size_lakhs
3158,5000,508.68,10.17


In [0]:
%sql
-- QUERY 1C: Portfolio Risk Metrics
--
-- BUSINESS PURPOSE:
-- Management wants to understand overall portfolio risk and health.
-- How many loans are healthy, how many are already defaulted, and how many are currently at risk?
--
-- WHAT TO CALCULATE:
-- Show overall portfolio metrics including:
-- Total loan volume
-- Healthy loan count
-- Defaulted loan count
-- Overdue loan count
--
-- Also calculate:
-- Overall default percentage
-- Overall portfolio-at-risk percentage
-- (At-risk means loans already failed OR showing payment stress)
--
-- EXPECTED INSIGHT:
-- Higher default percentage indicates weakening portfolio quality.
-- Higher at-risk percentage means more loans may move into default in future.
-- This helps identify whether the portfolio is stable or entering risk phase.
--
-- Write your query below:

In [0]:
%sql
SELECT
    COUNT(*) AS total_loans,

    SUM(CASE
            WHEN loan_status IN ('Active','Closed')
            THEN 1
            ELSE 0
        END) AS healthy_loan_count,

    SUM(CASE
            WHEN loan_status = 'Defaulted'
            THEN 1
            ELSE 0
        END) AS defaulted_count,

    SUM(CASE
            WHEN loan_status = 'Overdue'
            THEN 1
            ELSE 0
        END) AS overdue_count,

    ROUND(
        100.0 *
        SUM(CASE WHEN loan_status='Defaulted' THEN 1 ELSE 0 END)
        / COUNT(*),2
    ) AS default_pct,

    ROUND(
        100.0 *
        SUM(CASE
                WHEN loan_status IN ('Defaulted','Overdue')
                THEN 1
                ELSE 0
            END)
        / COUNT(*),2
    ) AS at_risk_pct

FROM workspace.edufin_static.loans;

total_loans,healthy_loan_count,defaulted_count,overdue_count,default_pct,at_risk_pct
5000,3476,1026,498,20.52,30.48


In [0]:
%sql
-- QUERY 1D: Financial Impact by Status
--
-- BUSINESS PURPOSE:
-- Decision-makers think in Crores, not loan counts.
-- "500 defaulted loans" is abstract. "₹8.5 Cr in defaulted loans" is real money.
-- This query translates operational numbers into financial language.
--
-- WHAT TO CALCULATE:
-- Show the financial value (in Crores) locked in each status.
-- How much money is in Active loans (earning interest)?
-- How much is in Defaulted loans (write-off candidates)?
-- How much is in Overdue loans (recovery pipeline)?
-- What's the combined "At Risk Value" (Defaulted + Overdue)?
--
-- EXPECTED INSIGHT:
-- If ₹12 Cr is at risk out of ₹500 Cr portfolio, that's 2.4% - manageable.
-- If ₹12 Cr is at risk out of ₹50 Cr portfolio, that's 24% - serious threat needing immediate action.

-- Write your query below:


In [0]:
%sql
SELECT
    loan_status,
    COUNT(*) AS loan_count,
    ROUND(SUM(loan_amount) / 10000000, 2) AS total_value_crores
FROM workspace.edufin_static.loans
GROUP BY loan_status

UNION ALL

SELECT
    'At Risk (Defaulted + Overdue)' AS loan_status,
    COUNT(*) AS loan_count,
    ROUND(SUM(loan_amount) / 10000000, 2) AS total_value_crores
FROM workspace.edufin_static.loans
WHERE loan_status IN ('Defaulted', 'Overdue')

ORDER BY total_value_crores DESC;

-- I grouped laons by status and calculated the financial value of each category to understand where portfolio capital is 
-- concentrated and how much money is exposed to risk.

--I expected Active loans to hold the majority of portfolio value, with Defaulted and Overdue loans representing
-- a smaller percentage of total exposure.

--Active loans account for 222.21 Cr while Closed loans account for 128.90 Cr. Defaulted loans represent 106.61 Cr and --Overdue loans represent 50.96 Cr. The combined at-risk value is 157.57 Cr.

-- Approximately 31% of the 508.68 Cr portfolio is tied up in Defaulted and Overdue loans. This suggests a substantial 
-- portfolio-wide risk rather than isolated credit issues and warrants immediate investigation into collections,
-- underwriting, and customer risk concentration.

loan_status,loan_count,total_value_crores
Active,2213,222.21
At Risk (Defaulted + Overdue),1524,157.57
Closed,1263,128.9
Defaulted,1026,106.61
Overdue,498,50.96


In [0]:
%sql
-- QUERY 1E: Executive Dashboard
--
-- BUSINESS PURPOSE:
-- Executives need ONE number that says "We're fine" or "We're in trouble."
-- This query consolidates all previous insights into a single executive-ready view with automated risk classification.
-- It's your "executive summary in a single row" - everything needed to make a go/no-go decision.
--
-- WHAT TO CALCULATE:
-- One comprehensive row showing:
-- All status counts and their percentages
-- Total portfolio financial metrics (values in Crores)
-- Automated health classification (HEALTHY / MODERATE RISK / HIGH RISK / CRITICAL) based on default rate thresholds
--
-- EXPECTED INSIGHT:
-- If health_status = "CRITICAL" and default_rate > 15%, immediate action required (capital raise, lending freeze, recovery acceleration).
-- If health_status = "MODERATE RISK" and default_rate = 8%, monitoring plan needed, not panic.
--
-- After running: Fill in the Executive Summary template below (Cell 8).

-- Write your query below:


In [0]:
%sql
SELECT
    COUNT(*) AS total_loans,

    SUM(CASE WHEN loan_status = 'Active' THEN 1 ELSE 0 END) AS active_loans,
    ROUND(
        SUM(CASE WHEN loan_status = 'Active' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS active_pct,

    SUM(CASE WHEN loan_status = 'Closed' THEN 1 ELSE 0 END) AS closed_loans,
    ROUND(
        SUM(CASE WHEN loan_status = 'Closed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS closed_pct,

    SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS defaulted_loans,
    ROUND(
        SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS default_pct,

    SUM(CASE WHEN loan_status = 'Overdue' THEN 1 ELSE 0 END) AS overdue_loans,
    ROUND(
        SUM(CASE WHEN loan_status = 'Overdue' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS overdue_pct,

    ROUND(SUM(loan_amount) / 10000000, 2) AS portfolio_value_crores,

    CASE
        WHEN
            SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) > 15
            THEN 'CRITICAL'
        WHEN
            SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) > 8
            THEN 'HIGH RISK'
        WHEN
            SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) > 3
            THEN 'MODERATE RISK'
        ELSE 'HEALTHY'
    END AS health_status

FROM workspace.edufin_static.loans;

total_loans,active_loans,active_pct,closed_loans,closed_pct,defaulted_loans,default_pct,overdue_loans,overdue_pct,portfolio_value_crores,health_status
5000,2213,44.26,1263,25.26,1026,20.52,498,9.96,508.68,CRITICAL


## Executive Summary (Query 1E Interpretation)

**Based on your Query 1E output, answer these:**

---

**1. Portfolio Health Status**

What health classification did Query 1E return (HEALTHY / MODERATE RISK / HIGH RISK / CRITICAL)? Does this match what you expected from Queries 1A-1D, or is there a surprise?



**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:** 
The portfolio health classification returned CRITICAL. This matches the findings from Queries 1A–1D and is not surprising. The default rate of 20.52% exceeds the 15% threshold established for critical risk. In addition, approximately ₹157.57 Crores of the portfolio value is tied to Defaulted and Overdue loans. These results indicate a portfolio-wide credit quality problem rather than isolated incidents involving a small number of borrowers.

**2. Default Rate Analysis**

Your Query 1E shows a default rate of _20.52__%. Is this acceptable for a fintech lender? Compare this to the 5%, 10%, 15% thresholds defined in Cell 1. What does this percentage actually mean for the business?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:** 
Query 1E shows a default rate of 20.52%.
This rate is significantly higher than all benchmark thresholds provided:
Healthy: Below 5%
Moderate Risk: 5%–10%
High Risk: 10%–15%
Critical: Above 15%
At 20.52%, more than one out of every five loans has defaulted. For a lending institution managing a portfolio worth ₹508.68 Crores, this level of default represents a serious threat to profitability, cash flow, and long-term sustainability. Immediate action is necessary to improve collections, review lending standards, and identify high-risk customer segments contributing to the losses.

**3. Financial Risk Exposure**

Query 1E shows ₹_157.57__ Cr at risk (Defaulted + Overdue combined). Out of a total portfolio of ₹_508.68__ Cr, this represents _30.98__% of the portfolio. Is this a liquidity crisis, a capital adequacy crisis, or a recoverable situation?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
Query 1E shows ₹157.57 Cr at risk (Defaulted and Overdue loans combined). Out of a total portfolio of ₹508.68 Cr, this represents approximately 30.98% of the portfolio value. This level of exposure is significant and cannot be considered normal for a healthy lending institution. While the company still has a substantial amount of performing loans, nearly one-third of the portfolio is at risk, creating pressure on cash flow, profitability, and future lending capacity. This appears to be more than a temporary liquidity issue and suggests a broader portfolio quality problem. Aggressive recovery efforts and a review of lending practices are recommended to prevent further deterioration.

**4. Recommended Action**

Based on the health status, default rate, and financial exposure above, what should be done in the next 48 hours? Be specific: freeze lending, accelerate recovery, raise capital, or continue monitoring?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
Based on the CRITICAL health classification, 20.52% default rate, and ₹157.57 Cr in at-risk exposure, immediate corrective action is recommended. In the next 48 hours, management should temporarily tighten lending approvals for higher-risk borrowers while launching an accelerated recovery and collections initiative focused on Defaulted and Overdue accounts. A detailed review of underwriting standards and customer risk profiles should also be conducted to identify the drivers of elevated defaults. Given that nearly one-third of the portfolio is at risk, continued monitoring alone would not be sufficient.

## INVESTIGATION SUMMARY

**Write a 3-4 sentence executive summary synthesizing your complete analysis:**

**Example format:**

*"Portfolio shows 12.4% default rate (MODERATE RISK) with ₹8.7 Cr at risk out of ₹70 Cr total. Risk concentrates in large loans (>₹6L) at 18% default rate. Root cause: lending policy failure post-Jan 2024 migration. Immediate action: freeze large loan approvals, launch ₹8.7 Cr recovery task force, audit migration data."*

---

**Your Summary (use your actual numbers):**
EduFin's portfolio is classified as CRITICAL, with a 20.52% default rate, significantly exceeding the 15% risk threshold. The company manages a ₹508.68 Cr portfolio, with ₹157.57 Cr (30.98%) tied up in Defaulted and Overdue loans. The analysis indicates a portfolio-wide credit quality issue rather than isolated borrower defaults. Immediate action should focus on strengthening collections and recovery efforts, reviewing underwriting standards, and tightening risk controls to prevent further portfolio deterioration.







---

### Revision History

**Track your daily refinements - delete the example row, keep only your actual changes:**

| Day | What Changed |
|-----|-------------|
| Day1 | Completed Portfolio Health investigation, calculated portfolio metrics, financial exposure, default rate analysis, executive dashboard, and business reasoning responses. |
| Day2 | |
| Day3 | |
| Day4 | |
| Day5 | |

## WHY GATE - Business Reasoning

### Question 1: At-Risk Definition

Your Query 1D calculates "Total At Risk Value". Did you include only Defaulted loans, or both Defaulted + Overdue? Explain why your choice is the correct business definition of "at risk" for a fintech lender in a crisis situation.



**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
I included both Defaulted and Overdue loans in the at-risk calculation. Defaulted loans represent money that has already failed to perform, while Overdue loans represent loans that are showing signs of distress and may eventually default if corrective action is not taken. In a fintech lending environment, management must evaluate both current losses and potential future losses. Therefore, the most appropriate business definition of "at risk" includes both Defaulted and Overdue loans because both categories expose the company to financial loss and require recovery efforts.

### Question 2: Crisis Classification

Look at your Query 1A (status distribution) and Query 1E (health classification). Which specific metric from these queries proves this is a systemic portfolio issue versus a temporary spike in defaults? What number would you cite to justify calling this a "crisis"?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
The strongest evidence that this is a systemic portfolio issue is the 20.52% default rate, which exceeds the 15% threshold established for a CRITICAL classification. In addition, Defaulted and Overdue loans account for approximately 30.98% of the portfolio value (₹157.57 Cr out of ₹508.68 Cr). These figures are too large to be explained by temporary fluctuations or isolated borrower problems. The combination of a high default rate, significant financial exposure, and CRITICAL health classification demonstrates a portfolio-wide credit quality crisis requiring immediate management attention.

### Question 3: Executive KPI Prioritization

If only one metric from this analysis had to be presented to leadership, which metric would you choose and why? Explain how this metric helps leadership evaluate portfolio health and business risk.

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**
The most important KPI is the 20.52% default rate because it summarizes portfolio quality in a single number that leadership can quickly understand and act upon. A default rate above the 15% critical threshold signals significant credit risk and directly impacts financial performance. While portfolio value and at-risk exposure are important supporting metrics, the default rate serves as the primary indicator of whether the lending business is healthy, deteriorating, or in crisis. For this reason, it should be the first metric presented to executives when evaluating portfolio health.

## SUBMISSION READINESS CHECK

**Before submitting, verify:**

✓ Data Validation (Cell 2) passed

✓ All 5 SQL queries (Cells 3-7) executed without errors

✓ Executive Summary (Cell 8) - all 4 questions answered

✓ Investigation Summary (Cell 9) - synthesis paragraph written, revision history updated

✓ WHY Gate (Cell 10) - all 3 questions answered with reasoning

✓ Financial values in Crores format (₹X.XX Cr)

✓ Percentages show 2 decimals (e.g., 11.78%)

✓ Deleted example rows from Revision History table

---

## SUBMISSION PROCESS

**Step 1:** Run Cell 12 below
- Enter your name and select day
- Get your submission filename and form link

**Step 2:** Download this notebook
- File → Export → Ipython notebook
- Rename the downloaded file as shown in Cell 12 output

**Step 3:** Submit via Google Form
- Use the form link from Cell 12 output
- Upload your ipynb file
- Note your entry number from form confirmation (tracking ID for feedback)

**Step 4:** Wait for feedback
- Expect feedback within 24 hours via WhatsApp

---

**Reminder:** Partial submissions accepted. If only 3 of 5 queries work, submit what you have with documentation of where you got stuck.


In [0]:
import re

FORM_LINK = "https://forms.gle/7xWVjeLRahSEMYXD9"

dbutils.widgets.removeAll()
dbutils.widgets.text("Name", "")
dbutils.widgets.text("StudentID", "")
dbutils.widgets.dropdown("day", "Day1", ["Day1", "Day2", "Day3"])

name = dbutils.widgets.get("Name")
batch = dbutils.widgets.get("StudentID")
day = dbutils.widgets.get("day")

if not name or not batch:
    print("⚠️  Enter your name and BatchNo above, then run this cell")
else:
    clean_name = re.sub(r'[^a-zA-Z0-9]', '', name.lower())
    filename = f"{clean_name}_{batch}_{day}.ipynb"
    
    print("File -> Export -> IPython notebook")
    print(f"Rename to: {filename}")
    print(f"Submit the file in the form link: {FORM_LINK}")
    print("\nPlease await for submission review.")

File -> Export -> IPython notebook
Rename to: reneekammeyer_SAP320B12_Day1.ipynb
Submit the file in the form link: https://forms.gle/7xWVjeLRahSEMYXD9

Please await for submission review.
